# COCO 2017 sur Kaggle

À lancer sur Kaggle, pas en local. FiftyOne télécharge COCO 2017 (train et validation).

Trois dossiers sont écrits dans `/kaggle/working/coco_out` :

- `vehicles_coco` : au plus 6 000 images qui ont une voiture, un bus, un camion ou une moto. Labels des 4 classes, mêmes identifiants que notre jeu (car 0, bus 1, truck 2, motorcycle 3).
- `coco_feux_positifs` : au plus 1 500 images qui ont un feu. Une seule classe `traffic_light`.
- `coco_negatifs_feux` : au plus 500 images qui ont un véhicule et aucun feu. Label vide.

Les annotations de feu sont chargées en même temps que les véhicules. Sans elles, une image avec un feu non annoté serait prise pour un négatif.

Aucun poids n'est téléchargé. Rien n'est entraîné ici. Le test court (200 images, 3 époques) est dans `kaggle_02_train.ipynb`, avec `SMOKE = True`.

In [ ]:
%pip install -q fiftyone
from pathlib import Path
import random
import shutil

import fiftyone as fo
import fiftyone.zoo as foz

SEED = 42
VEHICLE_CLASSES = ["car", "bus", "truck", "motorcycle"]
LIGHT_CLASS = "traffic light"
CLASS_TO_ID = {"car": 0, "bus": 1, "truck": 2, "motorcycle": 3}
LIMITS = {"vehicles_coco": 6000, "coco_feux_positifs": 1500, "coco_negatifs_feux": 500}
ZOO = {"train": 12000, "validation": 3000}
OUT = Path("/kaggle/working/coco_out")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)
fo.config.dataset_zoo_dir = "/kaggle/working/coco_zoo"

In [ ]:
def load_split(split, limit):
    name = f"coco_{split}"
    if fo.dataset_exists(name):
        fo.delete_dataset(name)
    return foz.load_zoo_dataset(
        "coco-2017",
        split=split,
        label_types=["detections"],
        classes=VEHICLE_CLASSES + [LIGHT_CLASS],
        max_samples=limit,
        shuffle=True,
        seed=SEED,
        dataset_name=name,
    )

datasets = [load_split(split, limit) for split, limit in ZOO.items()]

def boxes(sample, labels):
    detections = getattr(sample, "ground_truth", None)
    if detections is None:
        return []
    kept = []
    for det in detections.detections:
        if det.label not in labels:
            continue
        x, y, w, h = det.bounding_box
        if w <= 0 or h <= 0:
            continue
        kept.append((det.label, x, y, w, h))
    return kept

pool = []
for dataset in datasets:
    for sample in dataset:
        vehicle = boxes(sample, set(VEHICLE_CLASSES))
        lights = boxes(sample, {LIGHT_CLASS})
        if not vehicle and not lights:
            continue
        pool.append((sample.filepath, vehicle, lights))

rng = random.Random(SEED)
vehicles = [item for item in pool if item[1]]
positives = [item for item in pool if item[2]]
negatives = [item for item in pool if item[1] and not item[2]]
rng.shuffle(vehicles)
rng.shuffle(positives)
rng.shuffle(negatives)
chosen = {
    "vehicles_coco": vehicles[: LIMITS["vehicles_coco"]],
    "coco_feux_positifs": positives[: LIMITS["coco_feux_positifs"]],
    "coco_negatifs_feux": negatives[: LIMITS["coco_negatifs_feux"]],
}
for name, rows in chosen.items():
    if len(rows) < LIMITS[name]:
        raise SystemExit(f"{name} : {len(rows)} images, il en faut {LIMITS[name]}. Augmenter ZOO.")
print({name: len(rows) for name, rows in chosen.items()})

In [ ]:
def yolo_line(class_id, x, y, w, h):
    return f"{class_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"

def export(name, rows, mode):
    image_dir = OUT / name / "images"
    label_dir = OUT / name / "labels"
    image_dir.mkdir(parents=True)
    label_dir.mkdir(parents=True)
    for index, (source, vehicle, lights) in enumerate(rows):
        source_path = Path(source)
        stem = f"{index:05d}_{source_path.stem}"
        dest = image_dir / f"{stem}{source_path.suffix.lower()}"
        shutil.copy2(source_path, dest)
        lines = []
        if mode == "vehicles":
            lines = [yolo_line(CLASS_TO_ID[label], x, y, w, h) for label, x, y, w, h in vehicle]
        elif mode == "lights":
            lines = [yolo_line(0, x, y, w, h) for _label, x, y, w, h in lights]
        text = ("\n".join(lines) + "\n") if lines else ""
        (label_dir / f"{stem}.txt").write_text(text, encoding="utf-8")
    print(name, len(rows))

export("vehicles_coco", chosen["vehicles_coco"], "vehicles")
export("coco_feux_positifs", chosen["coco_feux_positifs"], "lights")
export("coco_negatifs_feux", chosen["coco_negatifs_feux"], "empty")
for dataset in datasets:
    dataset.delete()
print("écrit", OUT)